# EverFlavor AI - Recipe Photos: Food.com's Own Photos

**Team:** EverGlow | **Course:** ITAI 2277 | **Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: run 2026-10-25 and 2026-10-26.** Sections 1-3 and 5 ran here; sections 4, 6 and 7 ran through
> their scripts (each takes hours). Run it cell by cell. Section 2 needs a Kaggle login (a free API token in
> `~/.kaggle/kaggle.json`, or Colab's Kaggle secrets); sections 4, 6 and 7 are off by default (`DOWNLOAD_PHOTOS`, `CHECK_MATCHES`, `DISH_PHOTOS`).

---

## Purpose

About 222,000 of our 292,000 recipes have no photo: almost all are Food.com recipes, and the Food.com dataset we
use (shuyangli94, notebook 01) has no photos. Another Kaggle dataset of the same site, **Food.com - Recipes and
Reviews** (irkaal, **CC0: public domain**, checked through Kaggle's API on 2026-10-25), keeps each recipe's own
photo links in an `Images` column, under the same Food.com recipe numbers we use (`foodcom_<RecipeId>`).

| Section | What it does | Output |
|---|---|---|
| 2 | Downloads only that dataset's recipes file (not the 1.4 million reviews) | the kagglehub cache |
| 3 | Joins its first photo link to our Food.com recipes and counts how many get one | `data/processed/foodcom_photo_links.parquet` |
| 4 | Downloads the photos (paced, resumable), with the same logo fingerprint as the other photos | `data/raw/foodcom_photos/` |
| 5 | Hides logos and icons and writes what the app reads | `data/processed/foodcom_photos.parquet` |

Rules: open or research-use data only (CC0 here); every photo keeps a link to its recipe page; logos and photos
that do not show the dish are hidden (docs/plans/RECIPE_PHOTOS.md); nothing in `data/raw` is committed.

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook
%pip install -q pandas pyarrow requests pillow kagglehub

Note: you may need to restart the kernel to use updated packages.


In [1]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "cooking", "cooking_time", "dish_steps", "crew",
                      "cuisine", "diets", "dish_names", "environment", "evaluation", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline",
                      "progress", "recommend", "refresh", "reporting", "review", "safety", "seasons", "sources",
                      "stores", "variants"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

EVAL_DIR = Path("data/processed/agent_eval")   # one CSV per run, compared with the last one
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")

Runtime        : Local Jupyter (VS Code / Antigravity / JupyterLab)
Python         : 3.12.10
Project folder : ~\Downloads\jupiter-exploration\The-EverFlavor-AI


### Settings

`DOWNLOAD_PHOTOS` downloads about 200,000 photos (a few GB, a few hours at 4 requests at a time); leave it off to
only count the links. `PHOTO_SAMPLE` tries a small number first.

`CHECK_MATCHES` (section 6) and `DISH_PHOTOS` (section 7) are off by default: both run for hours and need the photo tools (`config/requirements-photos.txt`: PyTorch and open_clip; Colab already has PyTorch).


In [2]:
DOWNLOAD_PHOTOS = False      # section 4: download the photos themselves
PHOTO_SAMPLE = 200           # with DOWNLOAD_PHOTOS: download this many first (None for all)
CHECK_PHOTOS = True          # section 5: hide logos and icons, write what the app reads
CHECK_MATCHES = False        # section 6: does each photo show its dish? (CLIP, about 2 h on a laptop CPU)
DISH_PHOTOS = False          # section 7: free photos of the dish in general (Wikimedia Commons, hours)
SEARCH_ROUNDS = 1            # section 7: Commons search rounds of 2,000 dish names each
KAGGLE_DATASET = "irkaal/foodcom-recipes-and-reviews"
PHOTOS = PROJECT_ROOT / "data" / "raw" / "foodcom_photos"
LINKS = PROJECT_ROOT / "data" / "processed" / "foodcom_photo_links.parquet"

---

## 2. The Food.com photo links (CC0)

Only the recipes file is downloaded. Kaggle asks for a login: a free API token (kaggle.com > Settings > API >
Create New Token) saved as `~/.kaggle/kaggle.json`, or in Colab the `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets.

In [4]:
import kagglehub

recipes_file = Path(kagglehub.dataset_download(KAGGLE_DATASET, path="recipes.parquet"))
irkaal = pd.read_parquet(recipes_file, columns=["RecipeId", "Name", "Images"])
print(f"{len(irkaal):,} Food.com recipes in the CC0 dataset ({recipes_file.name})")

Extracting zip of recipes.parquet...


522,517 Food.com recipes in the CC0 dataset (recipes.parquet)


---

## 3. Join the photos to our recipes

A recipe's first photo link, matched by its Food.com number. The names are compared as a check that the numbers
mean the same recipe.

In [5]:
def first_photo(images):
    # The dataset keeps a list of links; some recipes have none
    if images is None or (hasattr(images, "__len__") and len(images) == 0):
        return None
    link = str(images[0]).strip().strip('"')
    return link if link.startswith("https://") else None


ours = pd.concat([pd.read_parquet(PROJECT_ROOT / "data" / "processed" / f"recipes_{split}.parquet",
                                  columns=["recipe_id", "recipe_name", "source"])
                  for split in ("train", "val", "test")])
ours = ours[ours["source"] == "foodcom"].copy()
ours["RecipeId"] = ours["recipe_id"].str.removeprefix("foodcom_").astype("int64")

irkaal["image_url"] = irkaal["Images"].map(first_photo)
links = ours.merge(irkaal[["RecipeId", "Name", "image_url"]], on="RecipeId", how="left")
same_name = links["Name"].str.lower().str.strip() == links["recipe_name"].str.lower().str.strip()
print(f"Our Food.com recipes: {len(ours):,}")
print(f"  found in the CC0 dataset: {links['Name'].notna().sum():,} ({links['Name'].notna().mean():.1%})")
print(f"  same name there: {same_name.sum():,} ({same_name.mean():.1%})")
print(f"  with a photo link: {links['image_url'].notna().sum():,} ({links['image_url'].notna().mean():.1%})")

LINKS.parent.mkdir(parents=True, exist_ok=True)
links[["recipe_id", "recipe_name", "image_url"]].dropna(subset=["image_url"]).to_parquet(LINKS, index=False)
print("Saved", LINKS.relative_to(PROJECT_ROOT))

Our Food.com recipes: 218,024
  found in the CC0 dataset: 217,716 (99.9%)
  same name there: 142,427 (65.3%)
  with a photo link: 111,803 (51.3%)
Saved data\processed\foodcom_photo_links.parquet


---

## 4. Download the photos (optional)

The same downloader as the other recipe photos (`scripts/recipe_photos/fetch.py`): 4 requests at a time, retries,
resumable, each file kept as downloaded, with an image fingerprint to find logos shared by many recipes. It has its
own folder and manifest, so the first photos' manifest is not touched.

Run on 2026-10-25 and 2026-10-26 with the script itself (`python -I scripts/recipe_photos/fetch.py data/raw/foodcom_links.csv data/raw/foodcom_photos`), since it takes hours: 111,614 of 111,803 photos downloaded (3.4 GB); 189 links no longer work (178 not found, 11 bad request). This cell was not run.

In [6]:
import subprocess

if DOWNLOAD_PHOTOS:
    table = pd.read_parquet(LINKS).rename(columns={"image_url": "url"}).assign(source="foodcom")
    if PHOTO_SAMPLE:
        table = table.sample(min(PHOTO_SAMPLE, len(table)), random_state=7)
    PHOTOS.mkdir(parents=True, exist_ok=True)
    links_csv = PHOTOS / "links.csv"
    table.to_csv(links_csv, index=False)
    subprocess.run([sys.executable, "-I", str(PROJECT_ROOT / "scripts" / "recipe_photos" / "fetch.py"),
                    str(links_csv), str(PHOTOS)], check=True)
else:
    print("DOWNLOAD_PHOTOS is off: only the links were counted.")

---

## 5. Hide logos and write what the app reads (optional)

Hidden: pictures shared by 3 or more recipes (logos) and files under 5 KB. The photo-matches-dish check
(`scripts/recipe_photos/check_photos.py`, CLIP) runs next on this folder's manifest; the app hides photos below its
cutoff (`PHOTO_CUTOFF` in `app/app.py`).

In [3]:
if CHECK_PHOTOS and (PHOTOS / "manifest.csv").exists():
    manifest = pd.read_csv(PHOTOS / "manifest.csv", low_memory=False)
    manifest = manifest[manifest["http_status"] == 200]
    shared = manifest.groupby("ahash")["recipe_id"].transform("nunique") >= 3
    manifest = manifest.assign(photo_ok=~shared & (manifest["bytes"] >= 5_000))
    out = manifest.drop_duplicates("recipe_id")[["recipe_id", "url", "photo_ok"]].rename(columns={"url": "image_url"})
    out.to_parquet(PROJECT_ROOT / "data" / "processed" / "foodcom_photos.parquet", index=False)
    print(f"{len(out):,} photos, {(~out['photo_ok']).sum():,} hidden as logos or icons")
else:
    print("CHECK_PHOTOS is off (or no photos downloaded yet).")

111,614 photos, 303 hidden as logos or icons


---

## 6. Does each photo show its dish? (optional)

A photo can be a logo, a drawing or a different dish. `scripts/recipe_photos/check_photos.py` compares each photo
with its recipe's name using CLIP (ViT-B/32, OpenAI weights, downloaded on first use): `embed` turns every photo into
a vector (cached in chunks, resumable), `score` compares it with the dish name. The app hides a photo when its
`clip_score` is below `PHOTO_CUTOFF` (-0.03, chosen on a hand-checked sample; `docs/plans/RECIPE_PHOTOS.md`).

Run on 2026-10-26 with the script itself: 111,613 Food.com photos scored in 110 minutes; **445 (0.4%) hidden**.
This cell was not run here.

In [ ]:
import importlib.util
import subprocess

if CHECK_MATCHES:
    if importlib.util.find_spec("open_clip") is None:   # the photo tools (Colab already has torch)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
    env = {**os.environ, "PHOTO_SET": "foodcom"}
    check = PROJECT_ROOT / "scripts" / "recipe_photos" / "check_photos.py"
    for step in ("embed", "score"):
        subprocess.run([sys.executable, "-I", str(check), step], check=True, env=env, cwd=PROJECT_ROOT)
    scores = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "photo_check_foodcom.parquet")
    hidden = int((scores["clip_score"] < -0.03).sum())
    print(f"{len(scores):,} photos scored; {hidden:,} would be hidden ({hidden / len(scores):.1%})")
else:
    print("CHECK_MATCHES is off.")

---

## 7. Photos of the dish in general, for recipes with no photo (Wikimedia Commons, optional)

For a recipe with no photo of its own, the app can show a free photo of the **dish in general** ("a photo of
chicken fajitas in general, not this recipe", with its credit). `scripts/dish_photos/fetch_dish_photos.py` finds them:

1. **Wikidata dishes:** recipe names are matched to Wikidata dishes, and each dish's own picture (property P18) is
   downloaded from Wikimedia Commons (stages `match`, `meta`, `download`, `curate`, `write`, then a CLIP `verify`).
2. **Commons search rounds** (`search --top 2000`): the most common dish names still without a photo are searched on
   Commons; a candidate is kept only if CLIP says it shows that dish by a clear margin (`clip_score` 0.06 or more,
   raised after round 1 from 0.045, on 40 hand-checked photos) and it does not look like a package, a landscape or
   a person. Each round takes 1-3 hours (most of it the downloads, 1 request a second).
3. **Review:** `data/reference/dish_photo_review.csv` turns a dish's photo down by hand (with the reason and who
   decided); the app reads it.

Only free licences are kept (CC0, public domain, CC BY, CC BY-SA, any version), every photo keeps its author and
licence, and every answer is cached, so a stopped run resumes. Results so far (2026-10-26, run with the script):
46,203 recipes have a dish photo (2,394 photos); with their own and Food.com's photos, **178,217 of 291,755
recipes (61.1%) show a photo**. The rest show the "Cooking up a picture..." skillet. This cell was not run here.

In [ ]:
if DISH_PHOTOS:
    if importlib.util.find_spec("open_clip") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
    dish = PROJECT_ROOT / "scripts" / "dish_photos" / "fetch_dish_photos.py"

    def stage(*args):
        subprocess.run([sys.executable, "-I", str(dish), *args], check=True, cwd=PROJECT_ROOT)

    for name in ("match", "meta", "download", "curate", "write"):   # 1. Wikidata dishes and their pictures
        stage(name)
    stage("verify", "--apply")                                       # CLIP check of those pictures
    for _ in range(SEARCH_ROUNDS):                                   # 2. Commons search rounds
        stage("search", "--top", "2000")
    stage("prune", "--apply")                                        # today's cutoff on earlier rounds too
else:
    print("DISH_PHOTOS is off.")

dish_file = PROJECT_ROOT / "data" / "processed" / "dish_photos.parquet"
if dish_file.exists():
    dish_photos = pd.read_parquet(dish_file)
    print(f"{dish_photos['recipe_id'].nunique():,} recipes have a photo of their dish in general "
          f"({dish_photos['image_file'].nunique():,} photos)")
    print(dish_photos["license"].value_counts().head(8).to_string())

---

## What comes next

- More Commons search rounds add fewer recipes each time (round 1: 9,385 recipes, then 7,284 after the stricter
  cutoff; round 2: 1,430), since the names left are a long tail. Other sources and a second check (Openverse,
  SigLIP 2, Gemini as a yes/no check) are compared in `docs/plans/RECIPE_PHOTOS.md`.
- The team checks the dish photos the way `data/reference/dish_photo_review.csv` does (keep = no, with a reason).
- Sources are recorded in `docs/sources.md`: Food.com - Recipes and Reviews (irkaal, CC0) and Wikimedia Commons.